# 11.20 Model-based RL & world models

Model-based RL buys sample efficiency by imagining consequences before acting.

A world model estimates transitions and rewards, then planning sweeps use imagined rollouts before spending more real samples. Save a copy to Drive to edit.

In [ ]:

import math
import random
import numpy as np
import matplotlib.pyplot as plt

SEED = 1119
random.seed(SEED)
np.random.seed(SEED)


def discounted_return(rewards, gamma=0.9):
    total = 0.0
    power = 1.0
    for reward in rewards:
        total = total + power * float(reward)
        power = power * gamma
    return total


lesson_rewards = [1, 0, 2]
lesson_gamma = 0.9
lesson_return = discounted_return(lesson_rewards, lesson_gamma)
assert abs(lesson_return - 2.62) < 1e-12


ACTIONS = [0, 1, 2, 3]
ACTION_NAMES = ["up", "right", "down", "left"]
DELTAS = {
    0: (-1, 0),
    1: (0, 1),
    2: (1, 0),
    3: (0, -1),
}


def make_grid_env(name, height, width, start, goal, walls=None, slip=0.0, wind=None, sparse=True, horizon=35, obs_noise=0.0):
    walls = set(walls or [])
    wind = wind or {}
    states = []
    state_index = {}
    for row in range(height):
        for col in range(width):
            cell = (row, col)
            if cell not in walls:
                state_index[cell] = len(states)
                states.append(cell)
    env = {
        "name": name,
        "height": height,
        "width": width,
        "start": start,
        "goal": goal,
        "walls": walls,
        "slip": float(slip),
        "wind": wind,
        "sparse": bool(sparse),
        "horizon": int(horizon),
        "states": states,
        "state_index": state_index,
        "n_states": len(states),
        "n_actions": 4,
        "obs_noise": float(obs_noise),
    }
    return env


def step_cell(env, cell, action, rng):
    actual = int(action)
    if rng.random() < env["slip"]:
        actual = int(rng.integers(0, 4))
    row_delta, col_delta = DELTAS[actual]
    row = cell[0] + row_delta
    col = cell[1] + col_delta
    pushed = env["wind"].get((cell[0], cell[1]), (0, 0))
    row = row + pushed[0]
    col = col + pushed[1]
    candidate = (max(0, min(env["height"] - 1, row)), max(0, min(env["width"] - 1, col)))
    if candidate in env["walls"]:
        candidate = cell
    reward = -0.02
    done = candidate == env["goal"]
    if done:
        reward = 1.0
    if not env["sparse"]:
        distance = abs(candidate[0] - env["goal"][0]) + abs(candidate[1] - env["goal"][1])
        reward = reward - 0.01 * distance
    return candidate, reward, done


def transition_table(env):
    n_states = env["n_states"]
    n_actions = env["n_actions"]
    transitions = np.zeros((n_states, n_actions, n_states), dtype=float)
    rewards = np.zeros((n_states, n_actions, n_states), dtype=float)
    rng = np.random.default_rng(SEED + env["n_states"])
    samples = 80
    for state_id, cell in enumerate(env["states"]):
        for action in range(n_actions):
            for _ in range(samples):
                next_cell, reward, done = step_cell(env, cell, action, rng)
                next_id = env["state_index"][next_cell]
                transitions[state_id, action, next_id] = transitions[state_id, action, next_id] + 1.0
                rewards[state_id, action, next_id] = rewards[state_id, action, next_id] + reward
            total = transitions[state_id, action].sum()
            mask = transitions[state_id, action] > 0
            rewards[state_id, action, mask] = rewards[state_id, action, mask] / transitions[state_id, action, mask]
            transitions[state_id, action] = transitions[state_id, action] / total
    return transitions, rewards


def build_f12_env_ladder():
    ladder = [
        make_grid_env("D1 two-state chain", 1, 2, (0, 0), (0, 1), horizon=6, sparse=False),
        make_grid_env("D2 slippery 3-state", 1, 3, (0, 0), (0, 2), slip=0.15, horizon=10, sparse=False),
        make_grid_env("D3 4x4 gridworld", 4, 4, (0, 0), (3, 3), walls={(1, 1), (2, 1)}, horizon=25, sparse=False),
        make_grid_env("D4 stochastic windy grid", 5, 5, (0, 0), (4, 4), walls={(1, 2), (2, 2)}, slip=0.2, wind={(3, 1): (-1, 0), (3, 2): (-1, 0)}, horizon=35, sparse=False),
        make_grid_env("D5 larger sparse grid", 7, 7, (0, 0), (6, 6), walls={(1, 1), (1, 2), (2, 4), (3, 1), (3, 2), (4, 4), (5, 2)}, slip=0.25, wind={(5, 3): (-1, 0)}, horizon=45, sparse=True),
    ]
    assert len(ladder) == 5
    assert [env["n_states"] for env in ladder] == [2, 3, 14, 23, 42]
    return ladder


def greedy_rollout(env, policy, episodes=20):
    rng = np.random.default_rng(SEED + env["height"] + env["width"])
    returns = []
    paths = []
    for episode in range(episodes):
        cell = env["start"]
        rewards = []
        path = [cell]
        for step in range(env["horizon"]):
            state_id = env["state_index"][cell]
            action = int(policy[state_id])
            cell, reward, done = step_cell(env, cell, action, rng)
            rewards.append(reward)
            path.append(cell)
            if done:
                break
        returns.append(discounted_return(rewards, lesson_gamma))
        paths.append(path)
    return float(np.mean(returns)), paths[-1]


def plot_grid_values(ax, env, values, title):
    grid = np.full((env["height"], env["width"]), np.nan)
    for cell, idx in env["state_index"].items():
        grid[cell] = values[idx]
    image = ax.imshow(grid, cmap="viridis")
    ax.set_title(title)
    ax.set_xticks([])
    ax.set_yticks([])
    return image


## The concept, built once: estimate $P$ and $R$, then plan\n\nThe model-based Bellman backup is\n$$V(s)=\max_a\sum_{s'}P(s'\mid s,a)\left(R(s,a,s')+\gamma V(s')\right).$$\nThe lesson's worked return remains $1+0.9\cdot0+0.9^2\cdot2=2.620$.

In [ ]:

def learn_model_and_plan(transitions, gamma=0.9, iterations=20):
    n_states = 2
    n_actions = 2
    counts = np.ones((n_states, n_actions, n_states)) * 1e-6
    reward_sums = np.zeros((n_states, n_actions, n_states))
    for state, action, reward, next_state in transitions:
        counts[state, action, next_state] = counts[state, action, next_state] + 1.0
        reward_sums[state, action, next_state] = reward_sums[state, action, next_state] + reward
    model_p = counts / counts.sum(axis=2, keepdims=True)
    model_r = reward_sums / np.maximum(counts, 1.0)
    values = np.zeros(n_states)
    for _ in range(iterations):
        q_values = np.sum(model_p * (model_r + gamma * values[None, None, :]), axis=2)
        values = np.max(q_values, axis=1)
    policy = np.argmax(q_values, axis=1)
    return model_p, model_r, values, policy


samples = [(0, 1, 1.0, 1), (0, 1, 1.0, 1), (1, 0, 0.0, 1)]
model_p, model_r, values, policy = learn_model_and_plan(samples)
print("P(s1|s0,right)", round(model_p[0, 1, 1], 6), "V0", round(values[0], 6))
assert abs(lesson_return - 2.62) < 1e-12
assert model_p.shape == (2, 2, 2)
assert int(policy[0]) == 1


The same method below learns a compact tabular model from samples, then performs planning sweeps over that model.

In [ ]:

def sample_model(env, samples_per_pair=12):
    rng = np.random.default_rng(SEED + env["n_states"] + samples_per_pair)
    n_states = env["n_states"]
    n_actions = env["n_actions"]
    counts = np.ones((n_states, n_actions, n_states)) * 1e-4
    reward_sums = np.zeros((n_states, n_actions, n_states))
    for state_id, cell in enumerate(env["states"]):
        for action in range(n_actions):
            for _ in range(samples_per_pair):
                next_cell, reward, done = step_cell(env, cell, action, rng)
                next_id = env["state_index"][next_cell]
                counts[state_id, action, next_id] = counts[state_id, action, next_id] + 1.0
                reward_sums[state_id, action, next_id] = reward_sums[state_id, action, next_id] + reward
    model_p = counts / counts.sum(axis=2, keepdims=True)
    model_r = reward_sums / np.maximum(counts, 1.0)
    return model_p, model_r


def plan_from_model(model_p, model_r, gamma=0.9, sweeps=60):
    n_states, n_actions, _ = model_p.shape
    values = np.zeros(n_states)
    for _ in range(sweeps):
        q_values = np.sum(model_p * (model_r + gamma * values[None, None, :]), axis=2)
        values = np.max(q_values, axis=1)
    return values, np.argmax(q_values, axis=1)


## The dataset ladder

In [ ]:

ladder = build_f12_env_ladder()
for env in ladder:
    sample_states = env["states"][:5]
    print(env["name"], "states", env["n_states"], "actions", env["n_actions"], "sample", sample_states)


In [ ]:

results = []
artifacts = []
for env in ladder:
    model_p, model_r = sample_model(env, samples_per_pair=16)
    values, policy = plan_from_model(model_p, model_r)
    mean_return, path = greedy_rollout(env, policy)
    results.append({"rung": env["name"], "return": mean_return})
    artifacts.append((env, values, path))
print("rung | return")
for row in results:
    print(row["rung"], round(row["return"], 3))


In [ ]:

fig, axes = plt.subplots(2, 5, figsize=(16, 6))
for col, (env, values, path) in enumerate(artifacts):
    plot_grid_values(axes[0, col], env, values, env["name"])
axes[1, 0].plot([idx + 1 for idx in range(len(results))], [row["return"] for row in results], marker="o")
axes[1, 0].set_title("return vs model rung")
axes[1, 0].set_xlabel("D rung")
axes[1, 0].set_ylabel("return")
for ax in axes[1, 1:]:
    ax.axis("off")
plt.tight_layout()


## Pitfall on D5: model bias compounds through rollouts

Planning too far with a stale biased model bootstraps from a moving target. Refreshing the model and using shorter planning rollouts reduces compounding bias.

In [ ]:

d5 = ladder[-1]
biased_p, biased_r = sample_model(d5, samples_per_pair=2)
biased_values, biased_policy = plan_from_model(biased_p, biased_r, sweeps=120)
biased_return, _ = greedy_rollout(d5, biased_policy)
refreshed_p, refreshed_r = sample_model(d5, samples_per_pair=20)
fixed_values, fixed_policy = plan_from_model(refreshed_p, refreshed_r, sweeps=40)
fixed_return, _ = greedy_rollout(d5, fixed_policy)
print("biased long-rollout return", round(biased_return, 3))
print("refreshed short-rollout return", round(fixed_return, 3))


## Evaluate it + Practice

- Metric: compare D1-D5 return against a no-skill baseline.
- Sanity check: D1 should solve by hand and match the exact assertion in the build-once cell.
- Ablation: turn off the key idea and verify the metric worsens on D4 or D5.
- Failure signal: curves that look good only because support, entropy, shape, or rollout bias was hidden.

Practice 1: change one ladder parameter and predict the metric direction before running.


Practice 2: reduce samples_per_pair and describe where the learned model becomes overconfident.